*   **解释**：梯度直接等于 **(预测概率 - 真实概率)**。
*   **效果**：
    *   如果预测错得离谱（ $ p_{\text{pred}} \approx 0, p_{\text{target}} = 1 $ ），梯度  $ \approx -1 $ ，很大！模型会迅速更新权重。
    *   如果预测很准（ $ p_{\text{pred}} \approx 1, p_{\text{target}} = 1 $ ），梯度  $ \approx 0 $ ，模型停止更新。
*   这种**误差越大，梯度越大**的线性关系，使得模型在训练初期收敛极快，避免了饱和区的梯度消失问题。

In [ ]:
import torch
import torch.nn as nn



rng = torch.Generator(device='cpu')
rng.manual_seed(42)
x = torch.multinomial(torch.tensor([0.1, 0.2, 0.3, 0.4]), num_samples=1, generator=rng)
print(x) # multinomial 直接输出索引
a = torch.randn(2,3,2)
print(a)
b = a[:,-1,:]

print(b)

topk = 2
torch.topk(b, min(b.shape[-1], topk), dim=-1)  #find topk values and indices along dim=-1

a = [1,2,3,4,5]
print(a[:-1])

a = torch.tensor(1,dtype=torch.float16)
print(a.dtype)
a = a.float()
print(a.dtype)


from torch.nn.functional import softmax


def corss_entropy(logits, targets, mask = -1):
    logits = softmax(logits, dim=-1)
    targets = targets.unsqueeze(-1) 
    props = logits.gather(dim = 1, index=targets)
    props = props.clamp(min=1e-10)
    return -props.log().mean()

def cross_entropy_stable(logits, targets):
    max_logits = logits.max(dim = -1)
    exp_prob = torch.exp(logits - max_logits.values.unsqueeze(1))
    sum_exp_prob = torch.sum(exp_prob, dim=-1, keepdim=True)
    log_prob = torch.log(sum_exp_prob)
    zy = logits.gather(dim = 1, index=targets.unsqueeze(1))
    loss = -zy + log_prob + max_logits.values.unsqueeze(1)
    loss = loss.mean()
    return loss

def cross_entropy_compelete(logits, targets, ignore_index = -1):
    max_logits = torch.max(logits, dim = -1, keepdim = True)[0]
    log_sum_prob = torch.log(torch.sum(torch.exp(logits - max_logits), dim = -1)) + max_logits.squeeze(-1)
    zy = logits.gather(dim = 1, index = targets.unsqueeze(1))
    sample_per_loss = -zy.squeeze(-1) + log_sum_prob

    if ignore_index is not None:
        mask = (targets == ignore_index)
        print(mask)
        sample_per_loss.masked_fill_(mask, 0.0)
        num_valid = (~mask).sum().float()

        if num_valid == 0:
            return torch.tensor(0.0, device=logits.device)
        else:
            return sample_per_loss.sum() / num_valid
    else:
        return sample_per_loss.mean()  



B, T, vocab_size = 2, 3, 12
logits = torch.randn(B, T, vocab_size)
targets = torch.randint(0, vocab_size, (B, T))
loss = corss_entropy01(logits.view(-1, logits.size(-1)), targets.view(-1))
print(loss)

loss = cross_entropy_stable(logits.view(-1, logits.size(-1)), targets.view(-1))
print(loss)

loss = nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=4)
print(loss)

loss = cross_entropy_compelete(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=4)
print(loss)

In [70]:
import torch
import torch.nn.functional as F
import math

def SlideWindowAttn(q,k,v,window_size):
    """
    滑动窗口注意力机制
    Args:
        q: 查询向量，形状为 (B, T, H, D)
        k: 键向量，形状为 (B, T, H, D)
        v: 值向量，形状为 (B, T, H, D)
        window_size: 窗口大小，一个整数或元组 (left, right)
    Returns:
        输出向量，形状为 (B, H, T, D)
    """
    window = window_size[0]
    Tk = k.size(1)
    Tq = q.size(1)
    q = q.transpose(1, 2)  # (B, T, H, D) -> (B, H, T, D)
    k = k.transpose(1, 2)  # 只改变逻辑读取，而不改变实际物理空间
    v = v.transpose(1, 2)
    
    if (window < 0 or window >= Tk) and Tk == Tq: # 训练阶段或初始填充(prefill)
        return Attention(q,k,v)
    if Tq == 1: # 解码阶段，每次只处理一个token
        k = k[:,:,-window - 1:,:]
        v = v[:,:,-window - 1:,:]
        return Attention(q,k,v)

    
    col_idx = Tk - Tq + torch.arange(Tq).unsqueeze(1)
    row_idx = torch.arange(Tk).unsqueeze(0)
    mask = row_idx <= col_idx

    print(mask)
    print(q.shape)
    print(k.shape)
    if window > 0 or window < Tk:
        mask = mask & (col_idx - row_idx <= window)
        print(mask)
        return ScaleDotProductAttention(q, k, v, attn_mask=mask)
        

def ScaleDotProductAttention(q,k,v, attn_mask=None):
    """
    缩放点积注意力机制
    Args:
        q: 查询向量，形状为 (B, H, T, D)
        k: 键向量，形状为 (B, H, T, D)
        v: 值向量，形状为 (B, H, T, D)
    Returns:
        输出向量，形状为 (B, H, T, D)
    """
    d = q.size(3)
    score = torch.matmul(q, k.transpose(2, 3)) / math.sqrt(d)

    if attn_mask is not None:
        score = score.masked_fill(attn_mask == 0, -float('inf'))
    weight = F.softmax(score, dim=-1)
    return torch.matmul(weight, v)

# q = torch.randn(1, 4, 2, 4)
# k = torch.randn(1, 10, 2, 4)
# v = torch.randn(1, 10, 2, 4)
# y = SlideWindowAttn(q,k,v,(1,4))


In [4]:
from datasets import load_dataset

# 配置
dataset_id = "karpathy/fineweb-edu-100b-shuffle"
split = "train"  # 通常只有 train 集
cache_dir = "./data_cache"  # 下载到的本地目录

print(f"开始下载数据集 {dataset_id} ...")
print("注意：全量数据非常大 (约 300GB+)，请确保磁盘空间充足。")
print("如果是测试，建议先只下载一部分 (见下方注释)。")

try:
    # 【全量下载】取消注释以下行以下载全部数据
    # dataset = load_dataset(dataset_id, split=split, cache_dir=cache_dir)
    
    # 【测试用：只下载前 2 个分片】
    # 如果你想先跑通流程，不要下全量，使用 streaming 模式取少量
    dataset = load_dataset(dataset_id, split=split, streaming=True, cache_dir=cache_dir)
    samples = []
    for i, item in enumerate(dataset):
        samples.append(item)
        # print(item['text'][:100])
        if i >= 100: break # 只取 100 条测试
    print(f"已获取 {len(samples)} 条样本用于测试")

    print("下载完成！")
    print(f"数据已缓存至: {cache_dir}")
    
    # 验证一下
    # print(f"数据集大小: {len(dataset)} 条记录")
    print(f"第一条数据预览: {dataset[0]['text'][:100]}...")

except Exception as e:
    print(f"下载出错: {e}")
    print("提示：如果网络不通，请尝试配置 HF_ENDPOINT 环境变量使用镜像站。")

开始下载数据集 karpathy/fineweb-edu-100b-shuffle ...
注意：全量数据非常大 (约 300GB+)，请确保磁盘空间充足。
如果是测试，建议先只下载一部分 (见下方注释)。
已获取 101 条样本用于测试
下载完成！
数据已缓存至: ./data_cache
第一条数据预览: <datasets.iterable_dataset.IterableColumn object at 0x1622ca1d0>...


In [81]:
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

def SlideWindowAttention(q,k,v,window_size):
    """
    滑动窗口注意力机制
    Args:
        q: 查询向量，形状为 (B, T, H, D)
        k: 键向量，形状为 (B, T, H, D)
        v: 值向量，形状为 (B, T, H, D)
        window_size: 窗口大小，一个整数或元组 (left, right)
    Returns:
        输出向量，形状为 (B, H, T, D)
    """
    window = window_size[0]
    print(window_size)
    Tk = k.size(1)
    Tq = q.size(1)
    q = q.transpose(1, 2)  # (B, T, H, D) -> (B, H, T, D)
    k = k.transpose(1, 2)  # 只改变逻辑读取，而不改变实际物理空间
    v = v.transpose(1, 2)
    
    if (window < 0 or window >= Tk) and Tk == Tq: # 训练阶段或初始填充(prefill)
        return ScaleDotProductAttention(q,k,v)
    if Tq == 1: # 解码阶段，每次只处理一个token
        k = k[:,:,-window - 1:,:]
        v = v[:,:,-window - 1:,:]
        return Attention(q,k,v)

    
    col_idx = Tk - Tq + torch.arange(Tq).unsqueeze(1)
    row_idx = torch.arange(Tk).unsqueeze(0)
    mask = row_idx <= col_idx

    print(mask)
    print(q.shape)
    print(k.shape)
    if window > 0 or window < Tk:
        mask = mask & (col_idx - row_idx <= window)
        print(mask)
        return ScaleDotProductAttention(q, k, v, attn_mask=mask)
        

def ScaleDotProductAttention(q,k,v, attn_mask=None):
    """
    缩放点积注意力机制
    Args:
        q: 查询向量，形状为 (B, H, T, D)
        k: 键向量，形状为 (B, H, T, D)
        v: 值向量，形状为 (B, H, T, D)
    Returns:
        输出向量，形状为 (B, H, T, D)
    """
    d = q.size(3)
    T = q.size(2)
    score = torch.matmul(q, k.transpose(2, 3)) / math.sqrt(d)

    if attn_mask is not None:
        score = score.masked_fill(attn_mask == 0, -float('inf'))
    else:
        col_i = torch.arange(T).unsqueeze(0)
        row_i = torch.arange(T).unsqueeze(1)
        mask = row_i <= col_i
        score = score.masked_fill(~mask, -float('inf'))

    weight = F.softmax(score, dim=-1)
    return torch.matmul(weight, v)

def ReLU2(x):
    return F.relu(x) ** 2

def has_ve(layer_id , layer_n):
    return (layer_id % 2 ) == (layer_n - 1) % 2

def norm(x):
    return F.rms_norm(x, (x.shape[-1],))

def apply_rotary_Position(x, cos, sin):
    B, T, H, D = x.shape
    x1 = x[..., :D//2]
    x2 = x[..., D//2:]
    x = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return x

def pre_compute_window_size(window_mode, config):
    mode  = window_mode.upper()
    # 只能是S或着L
    assert all(p in ['S', 'L'] for p in mode), "window_mode must be 'S' or 'L'"
    long_window = config.max_seq_len
    short_window = config.max_seq_len // 2
    char_to_window = {'S': (short_window,0), 'L': (long_window,0)}

    window_size = []
    for layer_idx in range(config.n_layer):
        window_size.append(char_to_window[mode[layer_idx % len(mode)]])
    
    window_size[-1] = (-1,0)
    return window_size

def precompute_rotary_embeddings(config, seq_len,base = 10000):
    head_dim = config.d_model // config.n_head
    inv_freq = 1.0 / (base **(torch.arange(0, head_dim, 2) / head_dim ))
    t = torch.arange(seq_len)
    freqs = torch.outer(t, inv_freq)
    cos = freqs.cos()
    sin = freqs.sin()
    cos = cos.bfloat16()
    sin = sin.bfloat16()
    cos = cos[None,:,None,:]
    sin = sin[None,:,None,:]
    return cos, sin

@dataclass
class GPTConfig:
    n_layer: int = 4
    n_head: int = 8
    d_model: int = 128
    vocab_size: int = 2024
    max_seq_len: int = 128
    ve_channel: int = 8
    window_mode: str = 'SSSL'


class CasualSelfAttention(nn.Module):
    def __init__(self, config, layer_id):
        super().__init__()
        self.config = config
        self.layer_id = layer_id
        self.c_q = nn.Linear(config.d_model, config.d_model)
        self.c_k = nn.Linear(config.d_model, config.d_model)
        self.c_v = nn.Linear(config.d_model, config.d_model)
        self.c_proj = nn.Linear(config.d_model, config.d_model)
        self.ve_gate = nn.Linear(config.ve_channel, config.n_head) if has_ve(layer_id, config.n_layer) else None

    def forward(self, x, ve, window_size, cos_sin, kv_cache = None):
        
        B, T, D = x.shape
        head_dim = D // self.config.n_head
        q = self.c_q(x) 
        k = self.c_k(x)
        v = self.c_v(x)

        q = q.view(B, T, self.config.n_head, head_dim)
        k = k.view(B, T, self.config.n_head, head_dim)
        v = v.view(B, T, self.config.n_head, head_dim)

        if kv_cache is not None:
            cos, sin = cos_sin
            T = 0
            cos = cos[:, T:]
            sin = sin[:, T:]

        if ve is not None and self.ve_gate is not None:
            ve = ve.view(B, T, self.config.n_head, head_dim)
            gata = 2 * torch.sigmoid(self.ve_gate(x[:, :, :self.config.ve_channel]))
            v = v + gata.unsqueeze(-1) * ve 

        cos, sin = cos_sin
        q = apply_rotary_Position(q, cos, sin)
        k = apply_rotary_Position(k, cos, sin)
        q, k = norm(q), norm(k)
        y = SlideWindowAttention(q, k, v, window_size[self.layer_id])
        y = y.contiguous().view(B, T, D)
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.c_fc = nn.Linear(config.d_model, config.d_model * 4)
        self.c_proj = nn.Linear(config.d_model * 4, config.d_model)

    def forward(self, x):
        x = self.c_fc(x)
        x = ReLU2(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    def __init__(self, config, layer_id):
        super().__init__()
        self.config = config
        self.attn = CasualSelfAttention(config, layer_id)
        self.mlp = MLP(config)

    def forward(self, x, ve, cos_sin, kv_cache = None):
        windows_size = pre_compute_window_size(self.config.window_mode, self.config)
        attn_out = self.attn(norm(x), ve, windows_size, cos_sin, kv_cache)
        x = x + attn_out
        mlp_out = self.mlp(norm(x))
        x = x + mlp_out
        return x

class GPT(nn.Module):
    def __init__(self, config, pad_to_vocab_size = 64):
        super().__init__()
        self.config = config
        head_dim = config.d_model // config.n_head
        self.rotary_len = config.max_seq_len * 10
        vocab_size = (config.vocab_size + pad_to_vocab_size - 1) // pad_to_vocab_size * pad_to_vocab_size
        self.transformer = nn.ModuleDict({
            'wte': nn.Embedding(vocab_size, config.d_model),
            'h' : nn.ModuleList([Block(config, layer_id) for layer_id in range(config.n_layer)])
        })
        self.lm_head = nn.Linear(config.d_model, vocab_size, bias=False)
        self.resid_lambdas = nn.Parameter(torch.ones(config.n_layer).bfloat16())
        
        self.x0_lambdas = nn.Parameter(torch.zeros(config.n_layer).bfloat16()) 
        self.value_embedding = nn.ModuleList([nn.Linear(config.d_model, config.d_model) for _ in range(config.n_layer)])

    @torch.no_grad()
    def init_weights(self):
        """
        Initialize the full model in this one function for maximum clarity.

        wte (embedding):     normal, std=1.0
        lm_head:             normal, std=0.001
        for each block:
            attn.c_q:        uniform, std=1/sqrt(n_embd)
            attn.c_k:        uniform, std=1/sqrt(n_embd)
            attn.c_v:        uniform, std=1/sqrt(n_embd)
            attn.c_proj:     zeros
            mlp.c_fc:        uniform, std=1/sqrt(n_embd)
            mlp.c_proj:      zeros
        """

        # Embedding and unembedding
        torch.nn.init.normal_(self.transformer.wte.weight, mean=0.0, std=1.0)  #正态分布
        torch.nn.init.normal_(self.lm_head.weight, mean=0.0, std=0.001)

        # Transformer blocks: uniform init with bound = sqrt(3) * std (same standard deviation as normal)
        n_embd = self.config.d_model
        s = 3**0.5 * n_embd**-0.5 # sqrt(3) multiplier makes sure Uniform achieves the same std as Normal
        for block in self.transformer.h:
            torch.nn.init.uniform_(block.attn.c_q.weight, -s, s) # weights use Uniform to avoid outliers
            torch.nn.init.uniform_(block.attn.c_k.weight, -s, s)  
            torch.nn.init.uniform_(block.attn.c_v.weight, -s, s)
            torch.nn.init.zeros_(block.attn.c_proj.weight) # projections are zero 投影层零初始化 
            torch.nn.init.uniform_(block.mlp.c_fc.weight, -s, s)
            torch.nn.init.zeros_(block.mlp.c_proj.weight)  # mlp投影层零初始化

        # Per-layer scalars
        self.resid_lambdas.fill_(1.0)   # 1.0 => typical residual connections at init
        self.x0_lambdas.fill_(0.1)      # 0.1 => small initial weight for skip connection to input embedding

        # Value embeddings (init like c_v: uniform with same std)
        for ve in self.value_embedding:
            torch.nn.init.uniform_(ve.weight, -s, s)  # value嵌入层均匀分布初始化和矩阵v保持一致

        # Gate weights init to zero so gates start at sigmoid(0) = 0.5, scaled by 2 -> 1.0 (neutral)
        for block in self.transformer.h:
            if block.attn.ve_gate is not None:
                torch.nn.init.zeros_(block.attn.ve_gate.weight)

        # Rotary embeddings
        head_dim = self.config.d_model // self.config.n_head
        cos, sin = precompute_rotary_embeddings(self.config, self.rotary_len)
        self.cos, self.sin = cos, sin

        # Cast embeddings to bf16: optimizer can tolerate it and it saves memory
        if self.transformer.wte.weight.device.type == "cuda":
            self.transformer.wte.to(dtype=torch.bfloat16)
            for ve in self.value_embedding:
                ve.to(dtype=torch.bfloat16)

    def forward(self, idx, target, kv_cache = None):
        x = self.transformer['wte'](idx)
        x = norm(x)
        T = x.shape[1]
        x0 = x
        T0 = 0 if kv_cache is None else kv_cache.get_pos()
        cos = self.cos[:,T0:T0 + T]
        sin = self.sin[:,T0:T0 + T]
        cos_sin = cos, sin
        for layer_id, block in enumerate(self.transformer['h']):
            x = self.resid_lambdas[layer_id] * x + self.x0_lambdas[layer_id] * x0
            ve = self.value_embedding[layer_id](x)
            x = block(x, ve, cos_sin)
        x = norm(x)
        logits = self.lm_head(x)
        logits = logits.float()
        logits = logits[..., :self.config.vocab_size]
        softcap = 15
        logits = torch.tanh(logits / softcap) * softcap

        if target is not None:
            loss = nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), target.view(-1), ignore_index=-1)
            return loss
        else:
            return logits

    def get_device(self):
        return self.transformer.wte.weight.device

    @torch.inference_mode()
    def generator(self, token, max_new_tokens, temperature=1.0, top_k=None, seed = 42):
        """
        Take a conditioning sequence of indices idx (LongTensor of shape (b,t)) and complete
        the sequence max_new_tokens times, feeding the predictions back into the model each time.
        Most likely you'll want to make sure to be in model.eval() mode of operation for this.
        """
        device = self.get_device()
        idx = torch.tensor(token, device=device, dtype = torch.long)
        rng = None
        if temperature > 0:
            rng = torch.Generator(device=device)
            rng.manual_seed(seed)
        
        for _ in range(max_new_tokens):
            logits = self.forward(idx)
            if top_k is not None and top_k > 0:
                logits = logits[:,-1,:]
                v, _ = torch.topk(logits,min(top_k, logits.size(-1)))  # 取top_k个最高概率的token，topk返回value和index
                logits[logits < v[:,-1,None]] = -float('Inf')  # 把其他token的logits设置为负无穷
            if temperature > 0:
                logits = logits / temperature
                probs = F.softmax(logits, dim=-1)
                idx_next = torch.multinomial(probs, num_samples = 1, generator = rng)
            else:
                idx_next = torch.argmax(logits, dim=-1, keepdim=True)  # 直接取最高概率的token
            idx = torch.cat((idx, idx_next), dim=1)  # 拼接新的token
            new_tokens = idx_next.item()
            yield new_tokens


gpt = GPT(GPTConfig())
gpt.init_weights()
batch_size = 4
seq_len = 8
vocab_size = 2024
idx = torch.randint(0, vocab_size, (batch_size, seq_len))
y0 = torch.randint(0, vocab_size, (batch_size, seq_len))
gpt(idx, y0)


(64, 0)
(64, 0)
(64, 0)
(-1, 0)


tensor(7.6093, grad_fn=<NllLossBackward0>)

In [ ]:
import torch

# 初始化权重（w=10）
w = torch.tensor([10.0], requires_grad=True)
# 梯度设为0（模拟梯度为0的极端场景）
w.grad = torch.tensor([0.0])

# 1. 原生Adam
optimizer_adam = torch.optim.Adam([w], lr=0.1, weight_decay=0)
optimizer_adam.step()
print(f"Adam更新后权重：{w.item():.2f}")  # 输出9.80（只减了0.2）

# 重置权重为10
w.data = torch.tensor([10.0])
w.grad = torch.tensor([0.0])

# 2. AdamW
optimizer_adamw = torch.optim.AdamW([w], lr=0.1, weight_decay=0.2)
optimizer_adamw.step()
print(f"AdamW更新后权重：{w.item():.2f}")  

Adam更新后权重：10.00
AdamW更新后权重：9.80


In [3]:
import os
# co-locate nanochat intermediates with other cached data in ~/.cache (by default)
if os.environ.get("NANOCHAT_BASE_DIR"):
    nanochat_dir = os.environ.get("NANOCHAT_BASE_DIR")
else:
    home_dir = os.path.expanduser("~")
    cache_dir = os.path.join(home_dir, ".cache")
    nanochat_dir = os.path.join(cache_dir, "nanochat")
os.makedirs(nanochat_dir, exist_ok=True)
print(f"nanochat_dir: {nanochat_dir}")

nanochat_dir: /Users/dongyong/.cache/nanochat
